In [32]:
from bs4 import BeautifulSoup
import re
import pandas as pd
from datetime import datetime, timedelta, timezone
from telethon import TelegramClient
import asyncio

In [24]:
#спарсим адреса телеграм-каналов
#используя html-разметку этой страницы https://tgstat.ru/ratings/channels/politics/public?sort=members
with open('page.html', 'r', encoding='utf-8') as f:
    soup = BeautifulSoup(f.read(), 'html.parser')
CHANNELS = []

for link in soup.find_all('a', href=True):
    if '/channel/@' in link['href']:
        channel = link['href'].split('/channel/')[1].split('/')[0]
        CHANNELS.append(channel)

In [25]:
len(CHANNELS)

100

In [18]:
API_ID = 
API_HASH = ""
SESSION = "tg_session"

In [31]:
async def scrape_last_2y(client, channels, days=730):
    cutoff = datetime.now(timezone.utc) - timedelta(days=days)
    rows = []
    
    for ch in channels:
        try:
            print(f"идет парсинг {ch}")
            entity = await client.get_entity(ch)
            
            async for m in client.iter_messages(entity, offset_date=cutoff, reverse=True):
                rows.append({
                    "channel": ch,
                    "message_id": m.id,
                    "date": m.date,
                    "text": m.message or "",
                    "views": m.views or 0
                })

        except ChannelPrivateError:
            print(f"канал {ch} приватный")
        except Exception as e:
            print(f"ошибка в канале {ch}: {e}")
    
    return pd.DataFrame(rows)

async def main():
    async with TelegramClient(SESSION, API_ID, API_HASH) as client:
        df = await scrape_last_2y(client, CHANNELS, days=730)
    df.to_parquet("tg_posts_2y.parquet", index=False)
    return df

In [34]:
if __name__ == "__main__":
    import nest_asyncio
    nest_asyncio.apply()
    df = asyncio.run(main())
    print(df.head())

идет парсинг @RKadyrov_95
идет парсинг @MDaudov_95
идет парсинг @medvedev_telegram
идет парсинг @adelimkhanov_95
идет парсинг @RVvoenkor
идет парсинг @rybar
идет парсинг @vv_volodin
идет парсинг @SolovievLive
идет парсинг @sanya_florida
идет парсинг @boris_rozhin
идет парсинг @dmitrynikotin
идет парсинг @obshina_ru
идет парсинг @mod_russia
идет парсинг @ejdailyru
идет парсинг @margaritasimonyan
идет парсинг @vysokygovorit
идет парсинг @MariaVladimirovnaZakharova
идет парсинг @mig41
идет парсинг @rlz_the_kraken
идет парсинг @vvgladkov
идет парсинг @SergeyKolyasnikov
идет парсинг @opersvodki
идет парсинг @nikitabsg
идет парсинг @Taynaya_kantselyariya
идет парсинг @russicaRU
идет парсинг @mnogonazi
идет парсинг @vatnoeboloto
идет парсинг @rian_novosti_russya


Telegram is having internal issues RpcCallFailError: Telegram is having internal issues, please try again later. (caused by GetHistoryRequest)


идет парсинг @strelkovii
идет парсинг @mos_sobyanin
идет парсинг @dimsmirnov175
идет парсинг @eschulmann
идет парсинг @aleksandr_skif
идет парсинг @voenkorKotenok
идет парсинг @stalin_gulag
идет парсинг @Putin_tramp_mobilizaciya_migrant
идет парсинг @slvn_pomet
идет парсинг @maester
идет парсинг @ToBeOr_Official
идет парсинг @news_kremlin
идет парсинг @generalsvr
идет парсинг @auantonov
идет парсинг @GardeZ66
идет парсинг @NeoficialniyBeZsonoV
идет парсинг @politadequate
идет парсинг @khazinml
идет парсинг @skabeeva
идет парсинг @dshrg2
идет парсинг @hellotransfer
идет парсинг @slutsky_l
идет парсинг @politjoystic
идет парсинг @Prigozhin_hat
идет парсинг @lentou4
идет парсинг @EvPanina
идет парсинг @AptiAlaudinovAKHMAT
идет парсинг @Hinshtein
идет парсинг @steven_bb
идет парсинг @jolynews
идет парсинг @ironlogica
идет парсинг @politkremlin
идет парсинг @vladislav_bytulka_novosty_video5
идет парсинг @mkhusnullin
идет парсинг @Sandymustache
идет парсинг @mardanaka
идет парсинг @vr_medins

In [35]:
df.shape

(1093965, 5)

In [41]:
df.to_csv('output_cursovaya.csv', index=False, encoding="utf-8-sig")

In [43]:
df1 = pd.read_csv('output_cursovaya.csv')

chunk_size = 1000000
chunks = [df1[i:i + chunk_size] for i in range(0, len(df1), chunk_size)]

for i, chunk in enumerate(chunks, 1):
    chunk.to_excel(f"output_cursovaya_part{i}.xlsx", index=False)
    print(f"Сохранена часть {i}: {len(chunk)} строк")

Сохранена часть 1: 1000000 строк
Сохранена часть 2: 93965 строк
